# KAE GPU Runner (Kaggle)

Обёртка для RFC 0022 Runner. Вся серверная логика — в репозитории
(`src/agents/runner/`), блокнот только поднимает окружение и запускает её.
Поэтому изменения в раннере не требуют правки блокнота: он клонирует репозиторий
при каждом запуске.

**Порядок:** клонировать репу → поставить зависимости → секреты в env →
поднять cloudflared → запустить раннер.

## Секреты (Settings → Add-ons → Secrets)

| Секрет | Назначение |
|---|---|
| `KAE_MANAGER_URL` | куда слать `/runner/announce` |
| `KAE_RUNNER_TOKEN` | Bearer-токен Manager↔Runner |

При push через `bin/push-kaggle-runner.sh` значения подставляются в
плейсхолдеры на месте; при ручном запуске берутся из Kaggle Secrets.

## Контракт

`POST /infer` — `{"task": "vision", "image_b64": "…"}` → `{"text": "…"}`,
как задано RFC 0022 §4.2. Страницу рендерит и присылает вызывающая сторона;
раннер только исполняет модель.


In [ ]:
# 1. Репозиторий и зависимости.
!git clone --depth=1 https://github.com/4stm4/BookAssembler.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip -q install fastapi 'uvicorn[standard]' transformers==4.49.0 qwen-vl-utils accelerate
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. Секреты → env.
# bin/push-kaggle-runner.sh подставляет значения вместо плейсхолдеров на
# push-time, чтобы они не попадали в git. Если push не использовался
# (ручной запуск в UI), читаем Kaggle Secrets.
import os

_URL = '__KAE_MANAGER_URL__'
_TOKEN = '__KAE_RUNNER_TOKEN__'
if _URL.startswith('__') or _TOKEN.startswith('__'):
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    _URL = sec.get_secret('KAE_MANAGER_URL')
    _TOKEN = sec.get_secret('KAE_RUNNER_TOKEN')

os.environ['KAE_MANAGER_URL'] = _URL
os.environ['KAE_RUNNER_TOKEN'] = _TOKEN
os.environ.setdefault('KAE_RUNNER_LOADERS', 'qwen_vl')
os.environ.setdefault('KAE_RUNNER_IDLE_TIMEOUT', '900')
print('manager:', _URL)
print('loaders:', os.environ['KAE_RUNNER_LOADERS'])

In [ ]:
# 3b. Диагностика GPU — до загрузки модели.
# Kaggle выдаёт разные карты, и установленный torch может не иметь ядер под
# доставшуюся архитектуру. Тогда модель займёт VRAM, но любой инференс упадёт
# с "no kernel image is available for execution on the device". Дешевле увидеть
# это здесь, чем по отказу первого запроса.
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader || echo 'нет nvidia-smi'
import torch
cc = torch.cuda.get_device_capability() if torch.cuda.is_available() else None
print('torch      :', torch.__version__)
print('cuda       :', torch.version.cuda)
print('устройство :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет CUDA')
print('capability :', f'sm_{cc[0]}{cc[1]}' if cc else '—')
print('собран под :', torch.cuda.get_arch_list() if torch.cuda.is_available() else '—')
if cc and f'sm_{cc[0]}{cc[1]}' not in torch.cuda.get_arch_list():
    print('\nВНИМАНИЕ: torch не содержит ядер под эту карту — инференс упадёт.')
    print('Перезапустите сессию, чтобы получить другую GPU.')

In [ ]:
# 4. cloudflared: публичный URL для раннера.
import itertools
import re
import subprocess

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5005'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
for line in itertools.islice(proc.stdout, 300):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
assert url, 'cloudflared did not report a public URL'
os.environ['KAE_RUNNER_PUBLIC_URL'] = url
print('Runner will announce as:', url)

In [ ]:
# 5. Раннер (foreground). Сам прогреет warmup-задачи, объявится Manager'у
#    и завершится после KAE_RUNNER_IDLE_TIMEOUT секунд простоя.
import sys
sys.path.insert(0, '/kaggle/working/repo')
!KAE_RUNNER_HOST=0.0.0.0 python -m src.agents.runner